## 1. Settings

In [2]:
# Colab-stable stack (no vLLM).
!pip install -q -U transformers accelerate bitsandbytes
!pip install -q -U transformers accelerate bitsandbytes hf_transfer
print("Installs done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 51.5 MB/s eta 0:00:00
Installs done


## 2. Config, Drive, schema



In [3]:
from __future__ import annotations
import json, os, re, random, math
from typing import Any, Dict, List, Optional, Tuple
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from google.colab import drive

try:                                   # accelerated Hugging Face downloads, if installed
    import hf_transfer
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
except Exception:
    pass

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then rerun."
print("GPU:", torch.cuda.get_device_name(0))

try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
    print("Hugging Face: authenticated.")
except Exception:
    print("Hugging Face: no token, continuing anonymously (fine for Qwen2.5).")

MODEL_ID       = "Qwen/Qwen2.5-3B-Instruct"
SEED           = 42
TARGET         = 10_000
ROWS_PER_RUN   = 5000          # NEW rows per run, then stop to inspect. Raise once happy; = TARGET for one shot.
GEN_BATCH      = 16
MAX_NEW_TOKENS = 480          # bio + comments + caption + audience note
TEMPERATURE    = 0.9
TOP_P          = 0.9
MAX_PROMPT_LEN = 1024
MAX_ATTEMPTS   = 8
USER_MSG       = "Generate the influencer text JSON now."

drive.mount("/content/drive")
OUT_DIR      = "/content/drive/MyDrive/BotOrNot"
os.makedirs(OUT_DIR, exist_ok=True)
OUTPUT_JSONL = os.path.join(OUT_DIR, "bon_dataset.jsonl")
OUTPUT_CSV   = os.path.join(OUT_DIR, "bon_dataset.csv")

random.seed(SEED)

COLUMNS = [
    "profile_id", "archetype", "is_fraud", "fraud_severity", "niche",
    "follower_count", "following_count", "posts_count", "account_age_months",
    "avg_likes", "avg_comments", "avg_views",
    "engagement_rate", "like_comment_ratio", "follower_following_ratio", "view_follower_ratio",
    "growth_spike_score", "audience_geo_mismatch", "audience_real_ratio",
    "comment_authenticity", "profile_pic_flag",
    "bio", "sample_comments", "post_caption", "audience_note",
]
MODEL_KEYS = ["bio", "comments", "post_caption", "audience_note"]
print("JSONL ->", OUTPUT_JSONL)

GPU: Tesla T4
Hugging Face: authenticated.
Mounted at /content/drive
JSONL -> /content/drive/MyDrive/BotOrNot/bon_dataset.jsonl


## 3. The fraud simulator
Each archetype has a fingerprint: ranges for follower scale, engagement rate, like/comment ratio,
follower/following ratio, growth spikiness, audience geo-mismatch, real-follower ratio, comment
authenticity, and profile-image flag. Ranges deliberately overlap so detection is non-trivial and
realistic. is_fraud and a 0-100 severity are derived from the archetype.

In [4]:
# fraud flag, severity band, and feature ranges per archetype.
# keys: fraud, sev, foll(log range), er, lcr, ffr, spike, geo, real, cauth, pic(prob)
ARCHE = {
 "authentic_organic":  dict(fraud=0, sev=(3,15),  foll=(30000,500000),  er=(2.0,6.0),  lcr=(25,70),  ffr=(3,30),  spike=(0.05,0.25), geo=(0.02,0.15), real=(0.90,0.99), cauth=(0.85,0.98), pic=0.0),
 "micro_real":         dict(fraud=0, sev=(3,15),  foll=(1000,25000),    er=(6.0,15.0), lcr=(15,50),  ffr=(1,8),   spike=(0.05,0.30), geo=(0.02,0.15), real=(0.88,0.99), cauth=(0.85,0.98), pic=0.0),
 "viral_legit":        dict(fraud=0, sev=(12,28), foll=(50000,1000000), er=(3.0,9.0),  lcr=(30,90),  ffr=(5,40),  spike=(0.60,0.95), geo=(0.05,0.20), real=(0.80,0.95), cauth=(0.75,0.95), pic=0.0),
 "lightly_boosted":    dict(fraud=0, sev=(22,40), foll=(20000,400000),  er=(1.5,4.0),  lcr=(40,120), ffr=(3,25),  spike=(0.30,0.60), geo=(0.10,0.35), real=(0.60,0.85), cauth=(0.60,0.85), pic=0.0),
 "bought_followers":   dict(fraud=1, sev=(55,85), foll=(50000,2000000), er=(0.2,1.2),  lcr=(30,100), ffr=(20,200),spike=(0.50,0.90), geo=(0.30,0.70), real=(0.20,0.55), cauth=(0.40,0.75), pic=0.0),
 "click_farm_audience":dict(fraud=1, sev=(55,88), foll=(40000,1500000), er=(0.3,1.5),  lcr=(20,80),  ffr=(10,120),spike=(0.30,0.80), geo=(0.50,0.90), real=(0.20,0.50), cauth=(0.30,0.60), pic=0.0),
 "bought_engagement":  dict(fraud=1, sev=(55,85), foll=(20000,500000),  er=(5.0,20.0), lcr=(80,250), ffr=(3,40),  spike=(0.20,0.60), geo=(0.10,0.40), real=(0.60,0.90), cauth=(0.15,0.50), pic=0.0),
 "engagement_pod":     dict(fraud=1, sev=(45,72), foll=(5000,150000),   er=(4.0,10.0), lcr=(10,40),  ffr=(1,10),  spike=(0.10,0.40), geo=(0.05,0.25), real=(0.75,0.95), cauth=(0.45,0.75), pic=0.0),
 "ai_persona":         dict(fraud=1, sev=(70,95), foll=(10000,800000),  er=(1.0,6.0),  lcr=(20,90),  ffr=(3,40),  spike=(0.30,0.80), geo=(0.20,0.60), real=(0.40,0.80), cauth=(0.30,0.60), pic=1.0),
}
ARCHETYPES = list(ARCHE.keys())

NICHES = ["fashion", "fitness", "beauty", "travel", "food", "gaming", "tech", "finance",
          "parenting", "home decor", "photography", "music", "comedy", "wellness", "sports"]

def _u(a, b): return random.uniform(a, b)
def _logu(a, b): return math.exp(random.uniform(math.log(a), math.log(b)))

def sample_features(archetype: str) -> Dict[str, Any]:
    p = ARCHE[archetype]
    foll = int(_logu(*p["foll"]))
    er = _u(*p["er"])                                   # engagement rate in percent
    ffr = _u(*p["ffr"])
    following = max(30, int(foll / ffr))
    posts = int(_logu(40, 3000))
    age = int(_u(6, 120))
    eng_total = er / 100.0 * foll                       # likes + comments
    lcr = _u(*p["lcr"])
    comments = max(1.0, eng_total / (1.0 + lcr))
    likes = max(1.0, eng_total - comments)
    views = foll * _u(0.2, 3.0)
    sev = round(min(100.0, max(0.0, _u(*p["sev"]) + random.gauss(0, 3))), 1)
    return {
        "follower_count": foll,
        "following_count": following,
        "posts_count": posts,
        "account_age_months": age,
        "avg_likes": int(likes),
        "avg_comments": int(comments),
        "avg_views": int(views),
        "engagement_rate": round(er, 2),
        "like_comment_ratio": round(likes / max(1.0, comments), 1),
        "follower_following_ratio": round(foll / max(1, following), 1),
        "view_follower_ratio": round(views / max(1, foll), 2),
        "growth_spike_score": round(_u(*p["spike"]), 3),
        "audience_geo_mismatch": round(_u(*p["geo"]), 3),
        "audience_real_ratio": round(_u(*p["real"]), 3),
        "comment_authenticity": round(_u(*p["cauth"]), 3),
        "profile_pic_flag": 1 if random.random() < p["pic"] else 0,
        "is_fraud": p["fraud"],
        "fraud_severity": sev,
    }

# comment/bio style per archetype (steers the LLM text)
STYLE = {
 "authentic_organic":  "genuine, specific comments that reference the actual content from real fans",
 "micro_real":         "warm, highly engaged genuine comments from a small loyal community",
 "viral_legit":        "a burst of excited genuine comments reacting to a post that blew up for real reasons",
 "lightly_boosted":    "mostly genuine comments with one or two generic ones mixed in",
 "bought_followers":   "a few normal comments that feel sparse next to a huge follower count",
 "click_farm_audience":"low-effort generic comments, some in unrelated languages, little topical relevance",
 "bought_engagement":  "generic praise and emoji spam like 'Nice!', 'Amazing', clearly low-effort bot-like",
 "engagement_pod":     "reciprocal promotional comments like 'love this, check out my page', 'support back'",
 "ai_persona":         "generic, slightly uncanny, non-specific comments that could apply to any post",
}

def make_job(archetype: str) -> Dict[str, Any]:
    return {"archetype": archetype, "niche": random.choice(NICHES), "features": sample_features(archetype)}

print("Archetypes:", len(ARCHETYPES))
print("Example job:", {k: v for k, v in make_job("bought_followers").items() if k != "features"})

Archetypes: 9
Example job: {'archetype': 'bought_followers', 'niche': 'photography'}


## 4. Prompt (bio + comments matching the archetype, bans AI tells)

In [4]:
BANNED = "Do NOT use em-dashes, en-dashes, or smart/curly quotes. Use plain hyphens and straight quotes only."

def build_system_prompt(job: Dict[str, Any]) -> str:
    a, niche = job["archetype"], job["niche"]
    return f"""You are a data-generation engine for an influencer-authenticity study. Your ONLY output is one valid JSON object. No markdown, no code fences, no commentary.

NICHE: {niche}
ACCOUNT TYPE (write text that fits this, do NOT name the type): {STYLE[a]}

Write realistic content for a FICTIONAL {niche} influencer: a profile bio, 4 sample comments left on their recent post, a caption for one of their recent posts, and a one-sentence note describing their audience. The comments must reflect the account type described above.

STYLE RULES:
- {BANNED}
- Sound like real social media text, not an AI. Keep the bio under 25 words, each comment under 15 words, the caption under 30 words, and the audience note under 20 words.
- Do not mention bots, fraud, or authenticity. Just write the bio and comments naturally.

Return ONLY this JSON object:
{{
  "bio": "the profile bio",
  "comments": ["comment 1", "comment 2", "comment 3", "comment 4"],
  "post_caption": "a caption for a recent post",
  "audience_note": "one sentence describing the audience"
}}"""

def to_prompt(system_msg: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system_msg},
         {"role": "user",   "content": USER_MSG}],
        tokenize=False, add_generation_prompt=True,
    )

## 5. JSON extraction + text sanitizer



In [5]:
_FENCE_RE = re.compile(r"`{3}(?:json)?\s*(.*?)\s*`{3}", re.DOTALL)

def sanitize(text: str) -> str:
    if not isinstance(text, str):
        return ""
    reps = {"—": "-", "–": "-", "‒": "-", "―": "-", "‘": "'", "’": "'", "“": '"', "”": '"', "…": "...", " ": " "}
    for a, b in reps.items():
        text = text.replace(a, b)
    text = re.sub(r":[a-z][a-z0-9_+\-]{1,20}:", "", text)   # strip emoji shortcodes like :salad:
    return re.sub(r"[ \t]+", " ", text).strip()

def extract_json(raw: str) -> Optional[dict]:
    raw = (raw or "").strip()
    cands = []
    m = _FENCE_RE.search(raw)
    if m:
        cands.append(m.group(1).strip())
    s, e = raw.find("{"), raw.rfind("}")
    if s != -1 and e > s:
        cands.append(raw[s:e + 1])
    cands.append(raw)
    for c in cands:
        try:
            obj = json.loads(c)
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            continue
    return None

## 6. Load the model (4-bit) and a batched generate helper


In [ ]:
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
print(f"Loading {MODEL_ID} in 4-bit ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                             device_map="auto", torch_dtype=torch.float16)
model.eval()

@torch.inference_mode()
def generate_batch(prompts: List[str]) -> List[str]:
    enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                    max_length=MAX_PROMPT_LEN).to(model.device)
    out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                         temperature=TEMPERATURE, top_p=TOP_P, pad_token_id=tokenizer.pad_token_id)
    gen = out[:, enc["input_ids"].shape[1]:]
    return tokenizer.batch_decode(gen, skip_special_tokens=True)

print("Model ready. VRAM:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

Loading Qwen/Qwen2.5-3B-Instruct in 4-bit ...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model ready. VRAM: 1.92 GB


## 7. Assemble a row + simulator sanity check

In [ ]:
def build_row(job: Dict[str, Any], data: dict, pid: str) -> Optional[Dict[str, Any]]:
    bio = sanitize(data.get("bio", ""))
    comments = data.get("comments", [])
    caption = sanitize(data.get("post_caption", ""))
    audience = sanitize(data.get("audience_note", ""))
    if not bio or not caption or not audience or not isinstance(comments, list) or len(comments) < 2:
        return None
    comments = [sanitize(str(c)) for c in comments if str(c).strip()]
    if len(comments) < 2:
        return None

    f = job["features"]
    row = {"profile_id": pid, "archetype": job["archetype"], "niche": job["niche"]}
    row.update(f)
    row["bio"] = bio
    row["sample_comments"] = " || ".join(comments)
    row["post_caption"] = caption
    row["audience_note"] = audience
    return row

# simulator sanity check: fraud archetypes should behave differently from authentic
import statistics as _st
_by = {a: [] for a in ARCHETYPES}
for _ in range(300):
    for a in ARCHETYPES:
        _by[a].append(sample_features(a)["engagement_rate"])
_auth = _st.mean(_by["authentic_organic"])
_bf   = _st.mean(_by["bought_followers"])
_be   = _st.mean(_by["bought_engagement"])
print(f"mean ER  authentic={_auth:.2f}  bought_followers={_bf:.2f}  bought_engagement={_be:.2f}")
assert _bf < _auth < _be, "engagement-rate fingerprints look wrong"
print("Simulator sanity check passed.")

mean ER  authentic=3.98  bought_followers=0.68  bought_engagement=12.06
Simulator sanity check passed.


## 8. Smart resume: scan existing JSONL

In [ ]:
def scan_existing(path: str):
    per = {a: 0 for a in ARCHETYPES}
    seen, total = set(), 0
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    o = json.loads(line)
                except json.JSONDecodeError:
                    continue
                total += 1
                if o.get("archetype") in per:
                    per[o["archetype"]] += 1
                seen.add(f"{o.get('archetype')}|{o.get('niche')}|{o.get('bio','')[:40]}")
    return total, per, seen

done, per_done, seen = scan_existing(OUTPUT_JSONL)
print(f"{done:,} rows already on Drive.")

5,000 rows already on Drive.


## 9. Balanced generation loop (small-batch, resumable)

Balanced across the 9 archetypes. Generates `GEN_BATCH` prompts at a time, adds up to `ROWS_PER_RUN`
new rows this run, requeues failures, and stops if it stalls. Re-run to add another batch.

In [ ]:
done, per_done, seen = scan_existing(OUTPUT_JSONL)   # always re-scan so ids and balance stay correct
if done >= TARGET:
    print(f"Target already reached: {done:,} rows.")
else:
    per_bucket = math.ceil(TARGET / len(ARCHETYPES))
    pending: List[str] = []
    for a in ARCHETYPES:
        pending += [a] * max(0, per_bucket - per_done.get(a, 0))
    random.shuffle(pending)
    pending = pending[: TARGET - done][: ROWS_PER_RUN]
    print(f"This run will add up to {len(pending):,} rows (done {done:,} / target {TARGET:,}).\n")

    counter = done
    stall = 0
    with open(OUTPUT_JSONL, "a", encoding="utf-8") as fh, \
         tqdm(total=TARGET, initial=done, unit="row", desc="Generating") as pbar:
        while pending:
            batch = pending[:GEN_BATCH]
            jobs = [make_job(a) for a in batch]
            prompts = [to_prompt(build_system_prompt(j)) for j in jobs]
            texts = generate_batch(prompts)

            produced, failed = 0, []
            for (a, job, text) in zip(batch, jobs, texts):
                data = extract_json(text)
                row = build_row(job, data, f"BON-{counter+1:05d}") if data else None
                if row is None:
                    failed.append(a); continue
                sig = f"{row['archetype']}|{row['niche']}|{row['bio'][:40]}"
                if sig in seen:
                    failed.append(a); continue
                seen.add(sig)
                counter += 1
                fh.write(json.dumps(row, ensure_ascii=False) + "\n")
                fh.flush()
                produced += 1
                pbar.update(1)

            pending = failed + pending[GEN_BATCH:]
            pbar.set_postfix({"retries": len(pending)})
            stall = stall + 1 if produced == 0 else 0
            if stall >= MAX_ATTEMPTS:
                print("\n[stop] too many empty steps in a row. Halting for inspection.")
                break

    total_now, _, _ = scan_existing(OUTPUT_JSONL)
    print(f"\nDone. Rows on Drive: {total_now:,}")

This run will add up to 5,000 rows (done 5,000 / target 10,000).



Generating:  50%|#####     | 5000/10000 [00:00<?, ?row/s]


Done. Rows on Drive: 10,000


## 9b. Inspect the batch (run after each small run)

In [ ]:
rows = [json.loads(l) for l in open(OUTPUT_JSONL, encoding="utf-8") if l.strip()]
print(f"Rows so far: {len(rows):,} / {TARGET:,}\n")

from collections import Counter
print("by archetype:", dict(Counter(r["archetype"] for r in rows)))
print("fraud share:", round(sum(r["is_fraud"] for r in rows) / max(1, len(rows)), 2))

tells = sum(1 for r in rows if re.search(r"[—–‘’“”]", r["bio"] + r["sample_comments"] + r.get("post_caption","") + r.get("audience_note","")))
print("rows with em-dash / smart quotes:", tells)

for r in random.sample(rows, min(3, len(rows))):
    print("\n" + "=" * 70)
    print(f"[{r['archetype']}]  fraud={r['is_fraud']}  severity={r['fraud_severity']}  niche={r['niche']}")
    print(f"  followers={r['follower_count']:,}  ER={r['engagement_rate']}%  geo_mismatch={r['audience_geo_mismatch']}  real_ratio={r['audience_real_ratio']}")
    print("  BIO:", r["bio"])
    print("  COMMENTS:", r["sample_comments"])
    print("  CAPTION:", r["post_caption"])

Rows so far: 10,000 / 10,000

by archetype: {'lightly_boosted': 1110, 'click_farm_audience': 1112, 'micro_real': 1111, 'bought_followers': 1111, 'engagement_pod': 1112, 'bought_engagement': 1110, 'ai_persona': 1111, 'authentic_organic': 1111, 'viral_legit': 1112}
fraud share: 0.56
rows with em-dash / smart quotes: 0

[engagement_pod]  fraud=1  severity=64.3  niche=home decor
  followers=5,610  ER=4.38%  geo_mismatch=0.236  real_ratio=0.812
  BIO: Niche: Home Decor - DIY enthusiast sharing cozy home ideas - Join the love! #homedecor - @HomeDecorJourney
  COMMENTS: Love this DIY project - check out my tips too! || Beautiful colors in your latest room - love it! || Wow, your new living room looks amazing! I'll definitely save these ideas. || This is exactly what I need - thanks for sharing!
  CAPTION: Springtime Refresh - New interior ideas to brighten your space! #springdecor - Stay tuned! 🌸✨

[bought_engagement]  fraud=1  severity=83.7  niche=photography
  followers=88,648  ER=11.08%  g

## 9c. so fresh and so CLEAN

In [ ]:
# One-time cleanup: re-sanitize existing rows (removes shortcodes from rows made before the fix)
import json, os
text_fields = ["bio", "sample_comments", "post_caption", "audience_note"]
rows = [json.loads(l) for l in open(OUTPUT_JSONL, encoding="utf-8") if l.strip()]
for o in rows:
    for k in text_fields:
        if isinstance(o.get(k), str):
            o[k] = sanitize(o[k])
tmp = OUTPUT_JSONL + ".tmp"
with open(tmp, "w", encoding="utf-8") as f:
    for o in rows:
        f.write(json.dumps(o, ensure_ascii=False) + "\n")
os.replace(tmp, OUTPUT_JSONL)          # atomic swap, safe if interrupted
print(f"Re-sanitized {len(rows)} rows. Now re-run cell 10 to refresh the CSV.")

Re-sanitized 1000 rows. Now re-run cell 10 to refresh the CSV.


## 9D. Touch-up

In [ ]:
# One-time repair: reassign unique sequential profile_ids
import json, os
rows = [json.loads(l) for l in open(OUTPUT_JSONL, encoding="utf-8") if l.strip()]
for i, o in enumerate(rows, start=1):
    o["profile_id"] = f"BON-{i:05d}"
tmp = OUTPUT_JSONL + ".tmp"
with open(tmp, "w", encoding="utf-8") as f:
    for o in rows:
        f.write(json.dumps(o, ensure_ascii=False) + "\n")
os.replace(tmp, OUTPUT_JSONL)
print(f"Reassigned {len(rows)} unique ids.")

Reassigned 1000 unique ids.


## 10. Export to CSV + quick checks

In [8]:
import pandas as pd

rows = []
with open(OUTPUT_JSONL, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue

df = pd.DataFrame(rows)[COLUMNS]
df.to_csv(OUTPUT_CSV, index=False)

print("Saved:", OUTPUT_CSV, "| shape:", df.shape)
print("\nRows per archetype:\n", df["archetype"].value_counts())
print("\nis_fraud balance:\n", df["is_fraud"].value_counts())
print("\nMean engagement_rate by archetype:")
print(df.groupby("archetype")["engagement_rate"].mean().round(2).sort_values())
tells = df["bio"].str.contains(r"[—–‘’“”]", regex=True).sum()
print("\nRows with em-dash / smart quotes in bio:", int(tells))
df.head(3)

Saved: /content/drive/MyDrive/BotOrNot/bon_dataset.csv | shape: (10000, 25)

Rows per archetype:
 archetype
click_farm_audience    1112
viral_legit            1112
engagement_pod         1112
bought_followers       1111
micro_real             1111
authentic_organic      1111
ai_persona             1111
lightly_boosted        1110
bought_engagement      1110
Name: count, dtype: int64

is_fraud balance:
 is_fraud
1    5556
0    4444
Name: count, dtype: int64

Mean engagement_rate by archetype:
archetype
bought_followers        3.96
click_farm_audience     4.25
lightly_boosted         5.17
ai_persona              5.56
authentic_organic       5.92
viral_legit             7.26
engagement_pod          7.40
micro_real              9.78
bought_engagement      10.41
Name: engagement_rate, dtype: float64

Rows with em-dash / smart quotes in bio: 0


,profile_id,archetype,is_fraud,fraud_severity,niche,follower_count,following_count,posts_count,account_age_months,avg_likes,...,view_follower_ratio,growth_spike_score,audience_geo_mismatch,audience_real_ratio,comment_authenticity,profile_pic_flag,bio,sample_comments,post_caption,audience_note
0,BON-00001,lightly_boosted,0,32.9,beauty,102332,1890,1297,85,2394,...,0.82,0.436,0.158,0.638,0.519,0,Beauty lover sharing tips - Follow for makeup ...,Great tips as always! || Love your skin! #beau...,Spring Skin Refresh - How to Achieve Radiant G...,My audience loves natural beauty and simple sk...
1,BON-00002,click_farm_audience,1,77.6,wellness,908685,13906,48,31,53881,...,1.82,0.654,0.702,0.494,0.577,0,Always sharing wellness tips in English & Span...,Great tips! #healthandfitness 💪🏼 || Love your ...,Starting my week right with a healthy smoothie...,My audience loves diverse wellness advice - bo...
2,BON-00003,micro_real,0,11.5,tech,1982,49,73,90,157,...,1.60,0.260,0.272,0.835,0.730,0,Tech enthusiast sharing daily tech tips & revi...,"Love your tech insights! 👍 || Great tips, alwa...",Check out our latest project - #AI advancement...,"Small, engaged community passionate about tech..."


## Make Some NOISE

In [5]:
# ============================================================
# Realistic Data: re-roll numeric features for realism.
# Keeps all text + labels, adds a case_type column.
# ============================================================
import json, os, random, math
import pandas as pd
random.seed(42)

FRAUDSET = [a for a, p in ARCHE.items() if p["fraud"]]
PRIMS = ["er", "lcr", "ffr", "spike", "geo", "real", "cauth"]
G = {k: (min(ARCHE[a][k][0] for a in ARCHE), max(ARCHE[a][k][1] for a in ARCHE)) for k in PRIMS}
BLEND, HARD_FRAC, HS, TRAP_FRAC = 0.30, 0.38, 0.60, 0.22   # tuned for observable AUC ~0.89

def _U(a, b): return random.uniform(a, b)
def _LU(a, b): return math.exp(random.uniform(math.log(a), math.log(b)))
def _primary(arch, b):
    p = ARCHE[arch]; return {k: (1 - b) * _U(*p[k]) + b * _U(*G[k]) for k in PRIMS}

def sample_hard(arch):
    p = ARCHE[arch]; prim = _primary(arch, BLEND); tag = "normal"; r = random.random()
    if p["fraud"] and r < HARD_FRAC:                       # sophisticated fraud hides its signature
        q = _primary("authentic_organic", BLEND); prim = {k: (1-HS)*prim[k] + HS*q[k] for k in prim}; tag = "masked_fraud"
    elif (not p["fraud"]) and r < TRAP_FRAC:               # clean account that looks suspicious
        q = _primary(random.choice(FRAUDSET), BLEND); prim = {k: (1-HS)*prim[k] + HS*q[k] for k in prim}; tag = "trap_clean"
    for k in PRIMS: prim[k] = min(G[k][1], max(G[k][0], prim[k]))
    foll = int(_LU(*p["foll"])); er = prim["er"]; ffr = max(1.0, prim["ffr"]); lcr = max(1.0, prim["lcr"])
    following = max(30, int(foll / ffr)); et = er / 100 * foll
    com = max(1.0, et / (1 + lcr)); lik = max(1.0, et - com); vw = foll * _U(0.2, 3.0)
    sev = round(min(100, max(0, _U(*p["sev"]) + random.gauss(0, 6))), 1)
    return {"is_fraud": p["fraud"], "fraud_severity": sev, "case_type": tag,
            "follower_count": foll, "following_count": following, "posts_count": int(_LU(40, 3000)),
            "account_age_months": int(_U(6, 120)), "avg_likes": int(lik), "avg_comments": int(com),
            "avg_views": int(vw), "engagement_rate": round(er, 2), "like_comment_ratio": round(lik/max(1, com), 1),
            "follower_following_ratio": round(foll/max(1, following), 1), "view_follower_ratio": round(vw/max(1, foll), 2),
            "growth_spike_score": round(prim["spike"], 3), "audience_geo_mismatch": round(prim["geo"], 3),
            "audience_real_ratio": round(prim["real"], 3), "comment_authenticity": round(prim["cauth"], 3),
            "profile_pic_flag": 1 if random.random() < p["pic"] else 0}

rows = [json.loads(l) for l in open(OUTPUT_JSONL, encoding="utf-8") if l.strip()]
for o in rows:
    o.update(sample_hard(o["archetype"]))     # replaces the numbers, keeps text + archetype

COLS = ["profile_id","archetype","is_fraud","fraud_severity","case_type","niche",
        "follower_count","following_count","posts_count","account_age_months","avg_likes","avg_comments","avg_views",
        "engagement_rate","like_comment_ratio","follower_following_ratio","view_follower_ratio","growth_spike_score",
        "audience_geo_mismatch","audience_real_ratio","comment_authenticity","profile_pic_flag",
        "bio","sample_comments","post_caption","audience_note"]
tmp = OUTPUT_JSONL + ".tmp"
with open(tmp, "w", encoding="utf-8") as f:
    for o in rows: f.write(json.dumps(o, ensure_ascii=False) + "\n")
os.replace(tmp, OUTPUT_JSONL)
pd.DataFrame(rows)[COLS].to_csv(OUTPUT_CSV, index=False)
print(f"Re-rolled {len(rows)} rows into realistic data. New column: case_type.")
print("case_type share:", dict(pd.Series([o['case_type'] for o in rows]).value_counts()))
print("Now re-run the EDA. No need to run cell 10.")

Re-rolled 10000 rows into realistic data. New column: case_type.
case_type share: {'normal': np.int64(6809), 'masked_fraud': np.int64(2141), 'trap_clean': np.int64(1050)}
Now re-run the EDA. No need to run cell 10.
